In [ ]:
from dotenv import load_dotenv
import os 

load_dotenv()

# Load base path
base_path = os.environ.get("BASE")
output_path = os.environ.get("OUTPUT")
input_path = os.environ.get("MAIN")

In [2]:
import pandas as pd 

claude_disclosure = pd.read_json(f"{output_path}claude/disclosure_3.jsonl", lines=True)
chatgpt_disclosure = pd.read_json(f"{output_path}chatgpt/disclosure_3.jsonl", lines=True)
gemini_disclosure = pd.read_json(f"{output_path}gemini/disclosure_3.jsonl", lines=True)
grok_disclosure = pd.read_json(f"{output_path}grok/disclosure_3.jsonl", lines=True)
deepseek_disclosure = pd.read_json(f"{output_path}deepseek/disclosure_3.jsonl", lines=True)

In [3]:
# Load base data: turns data 
import pandas as pd 

grok_turns     = pd.read_json(f'{base_path}/data/grok/grok_turns.jsonl', lines=True)
claude_turns   = pd.read_json(f'{base_path}/data/claude/claude_turns.jsonl', lines=True)
gemini_turns   = pd.read_json(f'{base_path}/data/gemini/gemini_turns.jsonl', lines=True)
deepseek_turns = pd.read_json(f'{base_path}/data/deepseek/deepseek_turns.jsonl', lines=True)
chatgpt_turns  = pd.read_json(f'{base_path}/data/chatgpt/chatgpt_turns.jsonl', lines=True)

In [4]:
# drop 'domain' to resolve confusion later on with sensitvity domain 
chatgpt_turns = chatgpt_turns.drop(columns=['domain'])

In [ ]:
from mapping import DOMAIN_MAP, FAMILY_MAP, ELICITATION_MAP, FUNCTION_MAP
import ast 
import json 
def parse(value):
    if isinstance(value, list):
        return [r for r in value if isinstance(r, dict)]
    if isinstance(value, dict):
        return [value]
    if isinstance(value, str) and value.strip():
        for loader in (json.loads, ast.literal_eval):
            try:
                return parse(loader(value))
            except Exception:
                pass
    return []


def apply_maps(value):
    out = []
    for r in parse(value):
        r = dict(r)
        for field, mapping, upper in [
            ("domain", DOMAIN_MAP, False),
            ("family", FAMILY_MAP, True),
            ("function", FUNCTION_MAP, True),
            ("elicitation", ELICITATION_MAP, True),
        ]:
            v = str(r.get(field) or "").strip()
            if v:
                v = mapping.get(v.lower(), v.upper() if upper else v.lower())
            r[field] = v
        r["sensitive"] = str(r.get("sensitive") or "").strip().upper()
        out.append(r)
    return out

# mapping is necessary sometimes when using LLMs for labeling (fixing orth mistakes, and so on)
for name in ["claude", "grok", "gemini", "deepseek", "chatgpt"]:
    df = globals()[f"{name}_disclosure"]
    df["data_per_turn"] = df["data_per_turn"].apply(apply_maps)

In [ ]:
import numpy as np
import statsmodels.formula.api as smf

frames = {
    "claude": (claude_disclosure, claude_turns),
    "grok": (grok_disclosure, grok_turns),
    "gemini": (gemini_disclosure, gemini_turns),
    "deepseek": (deepseek_disclosure, deepseek_turns),
    "chatgpt": (chatgpt_disclosure, chatgpt_turns),
}

turn_parts, ann_parts = [], []
for name, (disc, trn) in frames.items():
    t = trn.copy()
    t["assistant"] = name
    turn_parts.append(t)

    a = disc[["conversation_id", "data_per_turn"]].explode("data_per_turn").dropna(subset=["data_per_turn"])
    a = pd.concat(
        [a[["conversation_id"]].reset_index(drop=True), pd.DataFrame(list(a["data_per_turn"]))],
        axis=1,
    )
    a["assistant"] = name
    ann_parts.append(a)

turns = pd.concat(turn_parts, ignore_index=True)
anns = pd.concat(ann_parts, ignore_index=True)

for df in (turns, anns):
    df["turn_index"] = pd.to_numeric(df["turn_index"], errors="coerce")

keys = ["assistant", "conversation_id", "turn_index"]
anns = anns.dropna(subset=["turn_index"]).drop_duplicates(subset=keys)
turns = turns.dropna(subset=["turn_index"]).drop_duplicates(subset=keys)

# Create one master dataframe with all necessary details from labels and from original data
master = turns.merge(
    anns[keys + ["sensitive", "family", "domain", "function", "elicitation"]],
    on=keys, how="left",
)

master["annotated"] = master["sensitive"].isin(["YES", "NO"])
master["is_sensitive"] = master["sensitive"].eq("YES").astype("boolean")
master.loc[~master["annotated"], "is_sensitive"] = pd.NA

master = master.sort_values(["assistant", "conversation_id", "turn_index"]).reset_index(drop=True)
g = master.groupby(["assistant", "conversation_id"], sort=False)
master["turn_position"] = g.cumcount()
master["n_turns"] = g["turn_index"].transform("size")
master["relative_position"] = master["turn_position"] / (master["n_turns"] - 1).replace(0, np.nan)
master["log_n_turns"] = np.log(master["n_turns"])

ann = master[master["annotated"]].copy()
ann["is_sensitive"] = ann["is_sensitive"].astype(bool)
sens = ann[ann["is_sensitive"]].copy()

# Prevelance 

In [10]:
conv = ann.groupby(["assistant", "conversation_id"]).agg(
    n=("is_sensitive", "size"), k=("is_sensitive", "sum")
).reset_index()
conv["has"] = conv["k"] > 0

user = ann.groupby(["assistant", "user_id"]).agg(
    n=("is_sensitive", "size"), k=("is_sensitive", "sum")
).reset_index()
user["rate"] = user["k"] / user["n"]
user["any"] = user["k"] > 0

rng = np.random.default_rng(42)
rows = []
for a in sorted(ann["assistant"].unique()):
    c, u = conv[conv["assistant"] == a], user[user["assistant"] == a]

    ck, cn, ch = c["k"].to_numpy(float), c["n"].to_numpy(float), c["has"].to_numpy(float)
    cd = rng.integers(0, len(cn), size=(2000, len(cn)))
    boot_turn = ck[cd].sum(1) / cn[cd].sum(1)
    boot_conv = ch[cd].mean(1)

    ur = u["rate"].to_numpy()
    ud = rng.integers(0, len(ur), size=(2000, len(ur)))
    boot_user = ur[ud].mean(1)

    rows.append({
        "assistant": a,
        "turns": int(cn.sum()),
        "sensitive": int(ck.sum()),
        "turn_rate": ck.sum() / cn.sum(),
        "turn_low": np.percentile(boot_turn, 2.5),
        "turn_high": np.percentile(boot_turn, 97.5),
        "conv_rate": ch.mean(),
        "conv_low": np.percentile(boot_conv, 2.5),
        "conv_high": np.percentile(boot_conv, 97.5),
        "user_rate": ur.mean(),
        "user_median": np.median(ur),
        "users_any": u["any"].mean(),
    })

prevalence = pd.DataFrame(rows).set_index("assistant")
print(prevalence.round(4).to_string())

            turns  sensitive  turn_rate  turn_low  turn_high  conv_rate  conv_low  conv_high  user_rate  user_median  users_any
assistant                                                                                                                      
chatgpt    248158       6795     0.0274    0.0259     0.0289     0.0599    0.0578     0.0619     0.0207       0.0097     0.9500
claude      56436        838     0.0148    0.0115     0.0184     0.0256    0.0221     0.0289     0.0137       0.0000     0.4706
deepseek    32862        655     0.0199    0.0173     0.0228     0.0416    0.0377     0.0457     0.0150       0.0003     0.5000
gemini      90734       1291     0.0142    0.0128     0.0158     0.0278    0.0259     0.0296     0.0156       0.0043     0.6436
grok        52469       3854     0.0735    0.0611     0.0885     0.0800    0.0746     0.0858     0.0359       0.0128     0.7300


# Composition

In [11]:
FIELDS = ["family", "function", "elicitation", "domain"]

def composition(var):
    s = sens[sens[var] != ""]
    out = {"turn_pct": pd.crosstab(s["assistant"], s[var], normalize="index").mul(100).stack()}
    for level, tag in [("conversation_id", "conv"), ("user_id", "user")]:
        p = pd.crosstab([s["assistant"], s[level]], s[var])
        p = p.div(p.sum(axis=1), axis=0).mul(100)
        out[f"{tag}_mean"] = p.groupby("assistant").mean().stack()
        out[f"{tag}_median"] = p.groupby("assistant").median().stack()
    tab = pd.DataFrame(out).round(1)
    tab.index.names = ["assistant", var]
    return tab

for var in FIELDS:
    s = sens[sens[var] != ""]
    print(f"\n{var.upper()}   convos={s.groupby('assistant')['conversation_id'].nunique().to_dict()}"
          f"   users={s.groupby('assistant')['user_id'].nunique().to_dict()}")
    print(composition(var).to_string())


FAMILY   convos={'chatgpt': 2990, 'claude': 215, 'deepseek': 379, 'gemini': 771, 'grok': 713}   users={'chatgpt': 95, 'claude': 48, 'deepseek': 50, 'gemini': 65, 'grok': 73}
                      turn_pct  conv_mean  conv_median  user_mean  user_median
assistant family                                                              
chatgpt   DISCLOSURE      79.2       79.4        100.0       69.7         79.4
          OPINION          3.6        4.4          0.0        6.2          0.0
          REQUEST         17.3       16.1          0.0       24.1         13.8
claude    DISCLOSURE      89.7       76.8        100.0       73.4        100.0
          OPINION          0.7        2.8          0.0        1.7          0.0
          REQUEST          9.5       20.4          0.0       24.9          0.0
deepseek  DISCLOSURE      62.0       58.2        100.0       56.4         66.7
          OPINION         16.9       14.4          0.0       11.1          0.0
          REQUEST         21.1     

## Presistence and Onset

In [22]:
ann["prev"] = ann.groupby(["assistant", "conversation_id"])["is_sensitive"].shift(1)
pairs = ann.dropna(subset=["prev"])

persist = pairs.groupby(["assistant", "prev"])["is_sensitive"].agg(["size", "mean"]).unstack("prev")
persist.columns = [f"{s}_{'after_sens' if b else 'after_ord'}" for s, b in persist.columns]
print(persist.round(3))

first = sens.groupby(["assistant", "conversation_id"]).agg(
    first_turn=("turn_index", "min"), first_rel=("relative_position", "min")
).reset_index()
print("\n", first.groupby("assistant")[["first_turn", "first_rel"]].median().round(2))

cum = ann.groupby(["assistant", "conversation_id"])["is_sensitive"].cumsum() - ann["is_sensitive"]
risk = ann[cum == 0].copy()
risk["event"] = risk["is_sensitive"].astype(int)
risk["log_turn"] = np.log1p(risk["turn_position"])

hz = smf.logit("event ~ C(assistant) + log_turn", data=risk).fit(
    cov_type="cluster", cov_kwds={"groups": risk["user_id"]}, disp=False)
print("\nonset hazard")
print(np.exp(hz.params).round(3).to_string())

           size_after_ord  size_after_sens  mean_after_ord  mean_after_sens
assistant                                                                  
chatgpt            192748             5487           0.017            0.414
claude              47282              752           0.009            0.447
deepseek            23346              413           0.011            0.414
gemini              62156              814           0.008            0.484
grok                40115             3443           0.027            0.698

            first_turn  first_rel
assistant                       
chatgpt           1.0       0.33
claude            2.0       0.28
deepseek          0.0       0.29
gemini            1.0       0.50
grok              0.0       0.25

onset hazard
Intercept                   0.023
C(assistant)[T.claude]      0.353
C(assistant)[T.deepseek]    0.801
C(assistant)[T.gemini]      0.561
C(assistant)[T.grok]        1.528
log_turn                    0.658


## Models 

* m1: does the platform predict sensitivity, controlling for position and conversation length.
* m2: Controls for family, domain, and function so the comparison is "for the same kind of material, does this assistant draw it out more."

In [14]:
d = ann.dropna(subset=["relative_position", "user_id"]).copy()
d["y"] = d["is_sensitive"].astype(int)
m1 = smf.logit("y ~ C(assistant) + turn_position + relative_position + log_n_turns", data=d).fit(
    cov_type="cluster", cov_kwds={"groups": d["user_id"]}, disp=False)

print("odds of a sensitive turn")
print(pd.DataFrame({
    "OR": np.exp(m1.params), "low": np.exp(m1.conf_int()[0]),
    "high": np.exp(m1.conf_int()[1]), "p": m1.pvalues,
}).round(3))

e = sens[sens[["family", "domain", "function", "elicitation"]].ne("").all(1)].copy()
e["y"] = e["elicitation"].eq("ELICITED").astype(int)
m2 = smf.logit("y ~ C(assistant) + C(family) + C(domain) + C(function)", data=e).fit(
    cov_type="cluster", cov_kwds={"groups": e["user_id"]}, disp=False)

print("\nodds of elicited, content held constant")
print(pd.DataFrame({
    "OR": np.exp(m2.params), "low": np.exp(m2.conf_int()[0]),
    "high": np.exp(m2.conf_int()[1]), "p": m2.pvalues,
}).round(3).loc[lambda x: x.index.str.contains("assistant")])

odds of a sensitive turn
                             OR    low   high      p
Intercept                 0.014  0.007  0.027  0.000
C(assistant)[T.claude]    0.489  0.190  1.261  0.139
C(assistant)[T.deepseek]  0.711  0.360  1.405  0.327
C(assistant)[T.gemini]    0.564  0.299  1.064  0.077
C(assistant)[T.grok]      2.521  1.440  4.415  0.001
turn_position             0.999  0.999  1.000  0.011
relative_position         1.498  1.181  1.900  0.001
log_n_turns               1.215  1.075  1.372  0.002

odds of elicited, content held constant
                             OR    low   high      p
C(assistant)[T.claude]    1.467  1.005  2.141  0.047
C(assistant)[T.deepseek]  0.593  0.334  1.053  0.075
C(assistant)[T.gemini]    0.436  0.281  0.677  0.000
C(assistant)[T.grok]      1.011  0.784  1.303  0.936


/home/aelfraihi/.venv/lib/python3.13/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


# Topics Composition

In [15]:
import pandas as pd

# topic labels 
grok_topics     = pd.read_json(f'{input_path}/grok_topics.jsonl', lines=True)
claude_topics   = pd.read_json(f'{input_path}/claude_topics.jsonl', lines=True)
gemini_topics   = pd.read_json(f'{input_path}/gemini_topics.jsonl', lines=True)
deepseek_topics = pd.read_json(f'{input_path}/deepseek_topics.jsonl', lines=True)
chatgpt_topics  = pd.read_json(f'{input_path}/chatgpt_topics.jsonl', lines=True)

In [ ]:
topic_parts = []
for name in ["claude", "grok", "gemini", "deepseek", "chatgpt"]:
    t = globals()[f"{name}_topics"][["conversation_id", "conversation_label"]].copy()
    t["assistant"] = name
    topic_parts.append(t)

topics = pd.concat(topic_parts, ignore_index=True)
topics["topic"] = topics["conversation_label"].apply(
    lambda v: v[0] if isinstance(v, list) and v else (v if isinstance(v, str) else None)
)
topics = topics.dropna(subset=["topic"]).drop_duplicates(["assistant", "conversation_id"])

conv = ann.groupby(["assistant", "user_id", "conversation_id"]).agg(
    n=("is_sensitive", "size"), k=("is_sensitive", "sum")
).reset_index()
conv["has"] = conv["k"] > 0
conv = conv.merge(topics[["assistant", "conversation_id", "topic"]],
                  on=["assistant", "conversation_id"], how="left")

In [18]:
by_topic = conv.groupby("topic").agg(
    convs=("has", "size"), sens_rate=("has", "mean"), density=("k", "sum")
)
by_topic["density"] = by_topic["density"] / conv.groupby("topic")["n"].sum()
print(by_topic[by_topic["convs"] >= 50].sort_values("sens_rate", ascending=False).round(3).to_string())

                                 convs  sens_rate  density
topic                                                     
Sexual & Adult Content             297      0.747    0.465
Mental Health & Wellbeing          854      0.316    0.177
Relationships & Family            1732      0.286    0.156
Jailbreaking & Prompt Injection    546      0.233    0.176
Firearms & Weapons                 195      0.159    0.076
Creative Writing                  6366      0.124    0.068
Health & Fitness                  7174      0.100    0.058
AI Meta & Capabilities            2280      0.075    0.026
Law & Legal                       2255      0.071    0.030
Cybersecurity & Privacy            727      0.062    0.022
Politics & Current Affairs        1700      0.062    0.035
Image Generation & Editing        2859      0.056    0.031
Animals & Pets                    2289      0.053    0.032
Job Search & Career               2778      0.051    0.021
Religion & Philosophy             1205      0.043    0.0